# Feature Extraction — HOG + Color Histogram
**Kelompok 9 | COMP6577001 Machine Learning | BINUS University**

Pipeline: Image → Resize 64x64 → HOG (1764 dims) + Color Histogram HSV (96 dims) → 1860 dims per gambar

## 1. Setup

In [ ]:
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sys.path.append('../src')
from features import extract_features, preprocess_image, extract_hog_features, extract_color_histogram

DATA_DIR = Path('../data/raw/Dataset/archive/Fruits_Vegetables_Dataset(12000)')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(exist_ok=True)

SELECTED_ITEMS = ['Apple', 'Banana', 'Orange', 'Carrot', 'Cucumber', 'Potato', 'Tomato']
print('Setup done.')

## 2. Load Dataset Paths

In [ ]:
records = []
for category in ['Fruits', 'Vegetables']:
    for class_folder in sorted((DATA_DIR / category).iterdir()):
        if not class_folder.is_dir():
            continue
        name = class_folder.name
        if name.startswith('Fresh'):
            freshness, item = 'Fresh', name[5:]
        elif name.startswith('Rotten'):
            freshness, item = 'Rotten', name[6:]
        else:
            continue
        if item not in SELECTED_ITEMS:
            continue
        for img_path in class_folder.iterdir():
            if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                records.append({
                    'path': str(img_path),
                    'label': 0 if freshness == 'Fresh' else 1,
                    'freshness': freshness,
                    'item': item
                })

df = pd.DataFrame(records)
print(f'Total images: {len(df)}')
print(f'Fresh: {len(df[df["label"]==0])} | Rotten: {len(df[df["label"]==1])}')

## 3. Benchmark Waktu Feature Extraction (1 Gambar)

In [ ]:
sample_path = df['path'].iloc[0]

# Warmup
_ = extract_features(sample_path)

# Benchmark 10 runs
times = []
for _ in range(10):
    t0 = time.perf_counter()
    feat = extract_features(sample_path)
    t1 = time.perf_counter()
    times.append((t1 - t0) * 1000)

print(f'Feature vector shape : {feat.shape}')
print(f'Feature extraction   : {np.mean(times):.2f} ms (mean of 10 runs)')
print(f'Target               : < 70 ms')
print(f'Status               : {"✅ PASS" if np.mean(times) < 70 else "❌ FAIL — perlu optimasi"}')

## 4. Visualisasi HOG Feature

In [ ]:
from skimage.feature import hog
from skimage import exposure

sample_path = df['path'].iloc[0]
gray, hsv = preprocess_image(sample_path)

_, hog_img = hog(gray, orientations=9, pixels_per_cell=(8,8),
                 cells_per_block=(2,2), block_norm='L2-Hys',
                 visualize=True)
hog_img = exposure.rescale_intensity(hog_img, in_range=(0, 10))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
orig = cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB)
orig = cv2.resize(orig, (64, 64))

axes[0].imshow(orig)
axes[0].set_title('Original (64x64)')
axes[0].axis('off')

axes[1].imshow(gray, cmap='gray')
axes[1].set_title('Grayscale')
axes[1].axis('off')

axes[2].imshow(hog_img, cmap='gray')
axes[2].set_title('HOG Visualization')
axes[2].axis('off')

plt.suptitle('HOG Feature Extraction', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/feature_hog_viz.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: feature_hog_viz.png')

In [ ]:

# --- Color Histogram: Before vs After Background Removal ---
from features import _foreground_mask

fresh_path  = df[df['freshness'] == 'Fresh']['path'].iloc[0]
rotten_path = df[df['freshness'] == 'Rotten']['path'].iloc[0]

CH_NAMES  = ['H', 'S', 'V']
CH_COLORS = ['tab:red', 'tab:green', 'tab:blue']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for row, (path, label) in enumerate([(fresh_path, 'Fresh'), (rotten_path, 'Rotten')]):
    img_bgr = cv2.resize(cv2.imread(path), (64, 64))
    hsv     = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    mask    = _foreground_mask(hsv)
    fg_pct  = (mask == 255).mean() * 100

    axes[row, 0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    axes[row, 0].set_title(f'{label} — Original (64×64)')
    axes[row, 0].axis('off')

    for ch, (name, color) in enumerate(zip(CH_NAMES, CH_COLORS)):
        hist = cv2.calcHist([hsv], [ch], None, [32], [0, 256])
        axes[row, 1].plot(cv2.normalize(hist, hist).flatten(), color=color, label=name, alpha=0.85)
    axes[row, 1].set_title(f'{label} — Histogram (full image)')
    axes[row, 1].legend(); axes[row, 1].grid(True, alpha=0.3)

    for ch, (name, color) in enumerate(zip(CH_NAMES, CH_COLORS)):
        hist = cv2.calcHist([hsv], [ch], mask, [32], [0, 256])
        axes[row, 2].plot(cv2.normalize(hist, hist).flatten(), color=color, label=name, alpha=0.85)
    axes[row, 2].set_title(f'{label} — Histogram (BG removed, {fg_pct:.0f}% fg)')
    axes[row, 2].legend(); axes[row, 2].grid(True, alpha=0.3)

plt.suptitle('Color Histogram: Before vs After Background Removal', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/histogram_bg_removal_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: histogram_bg_removal_comparison.png')


## 5. Ekstrak Semua Fitur (Proses Seluruh Dataset)

In [ ]:
print(f'Memproses {len(df)} gambar...')
print('Estimasi waktu: ~2-5 menit')

X_list, y_list, failed = [], [], []
t_start = time.perf_counter()

for i, row in df.iterrows():
    try:
        feat = extract_features(row['path'])
        X_list.append(feat)
        y_list.append(row['label'])
    except Exception as e:
        failed.append(row['path'])
    if (len(X_list) + len(failed)) % 500 == 0:
        elapsed = time.perf_counter() - t_start
        print(f'  {len(X_list)} done | {elapsed:.1f}s elapsed')

t_total = time.perf_counter() - t_start
X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.int32)

print(f'\nSelesai dalam {t_total:.1f} detik')
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Failed : {len(failed)} gambar')
if failed:
    print('Failed paths:', failed[:5])

## 6. Simpan ke data/processed/

In [ ]:
np.save(PROCESSED_DIR / 'X.npy', X)
np.save(PROCESSED_DIR / 'y.npy', y)

print(f'Saved: X.npy  → {X.nbytes / 1024 / 1024:.1f} MB')
print(f'Saved: y.npy  → {y.nbytes / 1024:.1f} KB')
print(f'Path: {PROCESSED_DIR.resolve()}')
print()
print('=== SUMMARY ===')
print(f'Total samples   : {X.shape[0]}')
print(f'Feature dims    : {X.shape[1]} (HOG ~1764 + Color Hist 96)')
print(f'Fresh (label=0) : {(y==0).sum()}')
print(f'Rotten (label=1): {(y==1).sum()}')
print()
print('Lanjut ke 03_training_eval.ipynb')